# Zurich Tram Flow
**DELAY ANALYSIS AND PREDICTION · ZURICH TRAM NETWORK**

---


## Content

- [Facts](#facts)
- [Context](#context)
- [Deliverables](#deliverables)
- [Network](#network)
- [Dataset](#dataset)
- [Workflow](#workflow)


## Facts



| Feld | Wert |
|------|------|
| **Business-Frage** | Wo, wann und warum entstehen Verspätungen im Zürcher Tramnetz — und lassen sie sich vorhersagen? |
| **Stakeholder** | VBZ (Betreiber), Stadtplanung Zürich, Fahrgäste |
| **Methode** | EDA → Thematische Analyse (6 Bereiche) → Feature Engineering → ML-Modell |
| **Hauptdatenquelle** | VBZ IST-Daten 2023–2025 (opentransportdata.swiss) + GTFS + Meteo + Events |
| **Ziel-Metrik** | Vorhersagegenauigkeit (MAE) pro Linie/Stadtkreis; On-Time Performance (OTP) |
| **Out of Scope** | Echtzeit-Feed (GTFS-RT), Daten ab Format v2 (ab Mitte 2025), VBB Berlin |
| **Analysezeitraum** | 2023–2025 (IST-Daten Format v1, einheitlich) |
| **Stack** | Python · Polars · Pandas · LightGBM · Plotly · Jupyter |

## Context



### Scenario

Verspätungen im öffentlichen Nahverkehr sind ärgerlich — für Menschen und für das System.
Das Zürcher Tramnetz (VBZ) bietet eine außergewöhnlich gute Open-Data-Grundlage:
IST-Daten mit Echtzeit-Verspätungen pro Haltestelle, GTFS-Fahrplandaten, Wetterdaten
und Eventkalender — über drei Jahre (2023, 2024 und 2025).

Das Tram fährt im offenen Stadtverkehr — beeinflusst durch Autos, Fußgänger, Wetter,
Topografie und Großveranstaltungen. Der motorisierte Individualverkehr (MIV) ist dabei
einer der stärksten messbaren Faktoren: in Ferien und an Feiertagen fährt das Tram
nachweislich pünktlicher — weil weniger Autos auf der Strecke sind.

### Mission

Aufbau einer vollständigen Analyse- und Vorhersage-Pipeline für Verspätungen im
Zürcher Tramnetz — vom validierten Master-Datensatz bis zum interaktiven Dashboard.

### Zentrale Fragen

**Betrieb & Muster**
* Wo entstehen Verspätungen im Tramnetz — und zu welchen Zeiten?
* Welche Einflussfaktoren spielen die größte Rolle? (Wetter, Tageszeit, Events, Topografie, Autoverkehr)
* Lassen sich Verspätungen vorhersagen, bevor sie entstehen?

**Netz & Struktur**
* Hat der Netzausbau Dezember 2023 die Pünktlichkeit an den veränderten Linien und Stadtteilen verbessert oder verschlechtert?
* Welche Knotenpunkte sind kritische Hotspots und lösen Kettenreaktionen aus?
* Welche Stadtteile haben durch den Ausbau mehr oder weniger Anbindung bekommen?

**Systemisch**
* Was kann ein Betreiber oder eine Stadt konkret besser machen?


### Methode & Metriken

| Metrik | Zielwert | Begründung |
|--------|----------|-------------|
| On-Time Performance (OTP) | Baseline messen | Anteil Fahrten < 2 Min Verspätung |
| Mean Absolute Error (MAE) | < 60 Sek | Vorhersagegenauigkeit pro Linie/Stadtkreis |
| Hotspot-Ranking | Top-10 Haltestellen/Linien | Räumliche Delay-Konzentration nach Halt, Linie, Stadtkreis |
| Meteo-Effektstärke | Δ Delay pro Wetterbedingung | Schnee/Regen/Temperatur-Einfluss quantifiziert |
| Event Impact Score | Verspätungsanstieg messbar | Vergleich Event-Tage vs. normale Tage |

### Modellauswahl

Entscheidung für **LightGBM** (Gradient Boosting) — native Categorical Support, schnell auf großen Datensätzen (85M+ Zeilen), kein Overfitting durch Early Stopping. Kein Optuna — Feature Engineering war entscheidend, nicht Hyperparameter-Tuning.

#### Results

| Modell | Features | Test MAE | vs. Baseline |
|:---|:---|:---:|:---:|
| Stop Mean Baseline | — | 50.0s | — |
| **LightGBM v1** | 32 (Zeit · Wetter · Events · Linie · Stop) | 45.7s | −4.3s |
| **LightGBM v2** | 34 (+`prev_trip_delay`, +`stop_sequence_pct`) | **18,56 s** | **−31.4s (−63%)** |

Schlüsselerkenntnis: `prev_trip_delay` (Kaskadenindikator aus Analyse-Finding F-NET-07) ist das stärkste neue Feature — der Sprung von 45.7s auf 18,56 s kommt nicht vom Algorithmus, sondern vom Signal in den Daten.

→ Details: [`06_prediction_3-evaluation.ipynb`](06_prediction_3-evaluation.ipynb) · [`06_prediction_4-model_v2.ipynb`](06_prediction_4-model_v2.ipynb)

## Deliverables



| Artefakt | Beschreibung | Link |
|:---|:---|:---|
| **Interaktiver Report** | 3-Ebenen-Report: Scan · Dive · Deep-Dive | [`public/index.html`](../public/index.html) |
| **Dashboard** | Linie erkunden · Linien vergleichen · Delay vorhersagen | `uv run streamlit run apps/dashboard/app.py` |
| **Präsentation Management** | 24 Slides · Board-Version · "Vorhersagbar = Steuerbar" | [`public/presentation-v4.html`](../public/presentation-v4.html) |
| **Präsentation Technisch** | T-Shape · Data Engineering bis Modell | [`public/presentation.html`](../public/presentation.html) |
| **Live-Vorhersage Widget** | Stop × Linie × Stunde × Tagtyp × Wetter | [`public/live-prediction.html`](../public/live-prediction.html) |
| **Empfehlungskarte** | Risikomatrix Stop × Linie × Kontext | [`public/img/scheduling-recommendations-map.html`](../public/img/scheduling-recommendations-map.html) |
| **PDF Export** | Report als PDF | [`docs/exports/report.pdf`](../docs/exports/report.pdf) |


## Network

Das Zürcher Tramnetz (VBZ) besteht im Analysezeitraum 2023–2025 aus **16–18 Linien** je nach Fahrplanjahr.
Die GTFS-Daten werden jährlich als neue Version veröffentlicht — **j23**, **j24** und **j25** entsprechen den drei Betriebsjahren.

> **Interaktive Karte:** [`public/img/tram_lines_map.html`](../public/img/tram_lines_map.html)  
> Alle Linien mit offiziellen VBZ-Farben, Haltestellennamen und Streckenvergleich 2023 / 2024 / 2025.  
> Linien und Jahre können einzeln ein- und ausgeblendet werden.

### Fahrplanwechsel Dezember 2023 — j23 → j24

Der Fahrplanwechsel im Dezember 2023 war der **größte Netzausbau in der Geschichte der VBZ**
(*Tramnetz Süd*). Drei Linien wurden fundamental umgebaut:

| Linie | j23 Halte | j24 Halte | Veränderung | Neue Abschnitte |
| :---: | ---: | ---: | :--- | :--- |
| **9** | 24 | 32 | +8 Halte | Bellevue · Paradeplatz · Sihlstrasse · Goldbrunnenplatz |
| **11** | 20 | 33 | +13 Halte | Stadelhofen · Kreuzplatz · Burgwies · Rehalp |
| **13** | 11 | 30 | +19 Halte (+173%) | Altstetten · HB · Paradeplatz · Enge · Sihlcity Nord |
| **7** | 31 | 31 | 2 Halte umbenannt | Post Wollishofen → Renggerstrasse |
| **15** | 13 | 13 | 1 Halt umbenannt | Bucheggplatz → Bucheggplatz D |

Linien **10, 12, 14, 17** sind über alle drei Jahre identisch.
Linie **18** existiert nur im Fahrplanjahr 2024 (j24).

**Anbindungsveränderungen:** Durch den Ausbau gewannen vor allem **Kreis 3** (Sihlcity Nord, Enge) und **Kreis 8** (Rehalp, Burgwies, Kreuzplatz) an direkter Tramanbindung. Kreis 9 (Altstetten) erhielt erstmals eine durchgehende innenstadtnahe Verbindung via L13.

### Implications for Analysis

- **Linienvergleiche über Zeit:** Linien 9, 11 und 13 sind in j23 strukturell andere Linien als in j24/j25 — kürzere Strecken, weniger Halte, anderes Betriebsmuster. Direkter Jahresvergleich für diese Linien ist mit Vorsicht zu interpretieren.
- **Cancellation-Raten:** Erhöhte Ausfallraten in 2023 bei mehreren Linien können teilweise auf kürzere/andere Streckenführungen zurückzuführen sein — nicht zwingend auf schlechtere Betriebsqualität.
- **Feature Engineering:** `line_name` allein reicht nicht — das GTFS-Jahr (j23 vs. j24/j25) ist ein implizites Kontextmerkmal, das strukturelle Unterschiede kodiert.
- **Operative Qualität:** Der größte Netzumbau der VBZ-Geschichte hinterlässt **keinen erkennbaren Effekt** auf das Verspätungsverhalten — saisonale Muster dominieren, betriebliche Ereignisse nicht. Kein Einbruch, keine Einlaufzeit, keine Qualitätseinbußen. Die VBZ hat erhebliche Netzeingriffe operativ sauber abgewickelt — ein bemerkenswerter Befund.
- **`canceled`-Flag:** Netzweit erhöhte Rate Jan 2023 – Jun 2024, simultane Normalisierung Juli 2024 — wahrscheinlich eine Datendefinitions-Änderung beim Provider, nicht ein Infrastrukturproblem. Siehe Finding F-TARGET-05.


## Dataset




Die gesamte Data-Engineering-Phase wurde in einem separaten Research-Repo durchgeführt
und ist dort vollständig dokumentiert:

> **Quelle:** [`sf_data-research`](https://github.com/kaywiegand/sf_data-research)  
> **Status:** Phase 1 abgeschlossen — Datenbasis vollständig und validiert.

### Was wurde dort gemacht?

| Schritt | Beschreibung | Notebook |
| :--- | :--- | :--- |
| IST-Daten | Download 36 ZIP-Archive (38 GB), Filter auf VBZ & Tram, Parquet-Konvertierung | `vbz-ist-daten.ipynb` |
| GTFS | Fahrplandaten 2023–2025, Spatial Join Stadtkreise, Haltestellen-Lookup | `vbz-gtfs-data.ipynb` |
| Meteo | 3 Quellen konsolidiert (Stampfenbachstr. + Mythenquai), Stundenmittelwerte | `vbz-meteo-data.ipynb` |
| Events | 301 Einträge, 5 Kategorien, Gewichtungsschema 1–3 | `vbz-events-data.ipynb` |
| Benchmark | Polars vs. Pandas: 4× schneller, 4× weniger RAM | `vbz-pandas-vs-polars.ipynb` |
| Master-Merge | Left Join IST + GTFS + Meteo + Events → `vbz_master.parquet` | `vbz-data-master-preparation.ipynb` |
| Validierung | 8 Checks: Schema, Abdeckung, Wertebereiche, Nulls, Join-Qualität, Business-Logik | `vbz-data-master-validation.ipynb` |

### Wichtige Entscheidungen aus der Research-Phase

| Entscheidung | Was | Warum |
| :--- | :--- | :--- |
| Polars statt Pandas | Haupt-DataFrame-Bibliothek | 4× schneller, 4× weniger RAM bei 94 Mio. Zeilen |
| Left Join überall | Merge-Strategie | Kein Datenverlust durch Join-Lücken |
| 2024 als GTFS-Referenzjahr | Fahrplandaten | Vollständigste Datenlage, stabilstes Jahr |
| 2 Meteo-Stationen | Stampfenbachstrasse + Mythenquai | Zwei Topografien: Stadtlage vs. Seelage |
| Scope 2023–2025 v1 | Analysezeitraum | Einheitliches Datenformat, kein Mischformat |
| Stadtkreis im Lookup | district im GTFS-Join | Einmalig sauber im Master, kein wiederholter Spatial Join |
| Ausfälle behalten | `canceled = True` | Extremster Verspätungsfall, für Modell unverzichtbar |
| Schwellenwert Events | >1.000 Besucher | Kleinere Events kein messbarer Netzeinfluss |
| `trip_id` + `stop_sequence` | GTFS-Join-Erweiterung | Trip-Level-Analysen, Kaskadeneffekte, Hotspot-Erkennung |

### Data Volume

| Stufe | Menge |
| :--- | :--- |
| Rohdaten (schweizweit, komprimiert) | ~38 GB (36 ZIP-Archive) |
| Rohdaten entpackt | ~500–720 GB |
| Nach Filter VBZ + Tram (Parquet) | ~1,44 GB (1.096 Dateien) |
| Master-Datensatz | ~567 MB · 94 Mio. Zeilen · 26 Spalten |

### Data Dictionary

→ **[`docs/DATA_DICTIONARY.md`](../docs/DATA_DICTIONARY.md)** — 26 Spalten · Typen · Beschreibungen · Join-Strategie · Engineered Features

**Master-Datensatz:** `data/raw/zh-tram-data-master.parquet`  
~94.4 Mio. Zeilen · 26 Spalten · 4 Quellen: IST-Daten · GTFS · Meteo (3 Messstationen) · Events



### GTFS-Referenztabellen

**Verzeichnis:** `data/raw/gtfs/` — Referenzjahr 2024 (vollständigste Datenlage)

| Datei | Beschreibung | Verwendung |
| :--- | :--- | :--- |
| `gtfs_stops_lookup.parquet` | Haltestellen-Lookup: `bpuic` → `stop_name`, Koordinaten, `district_nr`, `district_name` | Join-Tabelle im Master |
| `gtfs_tram_stops.parquet` | Alle VBZ-Tram-Haltestellen mit Koordinaten | Geo-Visualisierungen |
| `gtfs_tram_routes.parquet` | Tramlinien (Route-ID, Linienname, Farbe) | Linien-Visualisierungen |
| `gtfs_tram_shapes.parquet` | Tram-Streckenverläufe als Koordinaten-Sequenzen | Streckenkarte |
| `gtfs_tram_trips.parquet` | Fahrten (Trip-ID, Route-ID, Shape-ID) | Verknüpfung Fahrten ↔ Strecken |
| `gtfs_zurich_stops.parquet` | Alle Zürich-Haltestellen (ZVV, nicht nur Tram) | Gesamtnetz-Überblick |
| `gtfs_zurich_routes.parquet` | Alle ZVV-Linien | Gesamtnetz-Überblick |
| `gtfs_zurich_shapes.parquet` | Alle ZVV-Streckenverläufe | Gesamtnetz-Karte |
| `gtfs_zurich_trips.parquet` | Alle ZVV-Fahrten | Gesamtnetz-Überblick |

## Workflow


### Projekt-Lifecycle

| | Data Engineering | Exploration | Preparation | Analysis | Modeling | Communication |
|:---|:---|:---|:---|:---|:---|:---|
| | Download & Filter | EDA Verteilungen | Cleaning | 6 Themenbereiche | Feature Engineering | Insights Report |
| | Data Joins | Integrität & Korrelationen | Train/Test Split | Hotspots · Temporal | LightGBM v1/v2 | Dashboard |
| | Validierung | Ausreisser Detection | Imputation | Meteo · Events | Kaskadenindikator | Präsentation |
| | Polars Benchmark | Key Hypotheses | Feature Export | 66 Findings | Dwell Simulator | Empfehlungskarte |

> **Data Engineering** abgeschlossen in [`sf_data-research`](https://github.com/kaywiegand/sf_data-research).  
> Vollständige Notebook-Übersicht → [README.md](../README.md)


### Konventionen

#### Variable Prefix

| Präfix | Typ | Bedeutung |
|:---|:---|:---|
| `lf_` | `pl.LazyFrame` | Noch nicht im RAM — Operationen werden zu einem Scan zusammengefasst |
| `df_` | `pl.DataFrame` | Nach `.collect()` — vollständig im RAM |

> **Regel:** `lf_` solange die Pipeline aufgebaut wird. Einmalig `.collect()` → ab dann `df_`.

---

#### Variables and Files Across All Notebooks

| Variable / Datei | Notebook | Beschreibung |
|:---|:---|:---|
| `lf_raw` | 01 · 02 | `pl.scan_parquet(master)` — Rohdaten, lazy |
| `df_eda` | 01 | Sample für EDA: `lf_raw.gather_every(n).collect()` (~1 Mio. Zeilen) |
| `lf_clean` | 02 | `structural_cleaning_pipeline(lf_raw)` — lazy vor dem Split |
| `train_raw.parquet` / `test_raw.parquet` | 02 → `data/interim/` | Temporal Split: 2023–2024 Train / 2025 Test |
| `train_prepared.parquet` / `test_prepared.parquet` | 02 → `data/processed/` | Nach Meteo-Imputation (Forward/Backward Fill) |
| `train_features.parquet` / `test_features.parquet` | 02 → `data/processed/` | Nach Zeit- und Wetter-Features |
| `lf` / `lf_all` / `lf_clean` | 03_analysis_* | Via `setup_analysis()` — scannt `train_final` / `test_final` |
| `train_final.parquet` / `test_final.parquet` | 05 → `data/processed/` | Finales Feature-Set: 55.5 Mio. Zeilen · 40 Spalten |